# Walmart Analysis part 2

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

user = "root"
password = getpass("enter mysql password: ")
host = "localhost"
port = 3306
database = "walmart_db"

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=user,
    password=password,
    host=host,
    port=port,
    database=database
)

engine = create_engine(connection_url)

query = """
select database() as current_database;
"""

pd.read_sql(query, engine)


enter mysql password:  ········


,current_database
0,walmart_db


In [2]:
query = """
select database() as current_database;
"""

result = pd.read_sql(query, engine)
result

,current_database
0,walmart_db


In [3]:
query = """
select * from walmart_sales;
"""
result = pd.read_sql(query,engine)
result

,transaction_id,customer_id,product_id,product_name,category,quantity_sold,unit_price,transaction_date,store_id,store_location,...,actual_demand,revenue,forecast_error,transaction_year,transaction_month,transaction_quarter,forecast_error_percentage,inventory_gap,understock_flag,reorder_required
0,1,2824,843,Fridge,Electronics,3,188.46,2024-03-31 21:46:00,3,"Miami, FL",...,179,565.38,7,2024,3,1,3.91,67,0,0
1,2,1409,135,TV,Electronics,4,1912.04,2024-07-28 12:45:00,5,"Dallas, TX",...,484,7648.16,375,2024,7,3,77.48,-441,1,1
2,3,5506,391,Fridge,Electronics,4,1377.75,2024-06-10 04:55:00,1,"Los Angeles, CA",...,416,5511.00,127,2024,6,2,30.53,-5,1,0
3,4,5012,710,Smartphone,Electronics,5,182.31,2024-08-15 01:03:00,5,"Miami, FL",...,446,911.55,272,2024,8,3,60.99,6,0,0
4,5,4657,116,Laptop,Electronics,3,499.28,2024-09-13 00:45:00,6,"Chicago, IL",...,469,1497.84,182,2024,9,3,38.81,-57,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,6898,852,Headphones,Appliances,1,682.15,2024-07-08 06:13:00,17,"New York, NY",...,294,682.15,37,2024,7,3,12.59,-219,1,1
4996,4997,8412,886,Laptop,Appliances,3,1418.09,2024-02-07 11:30:00,16,"Los Angeles, CA",...,397,4254.27,9,2024,2,1,2.27,-190,1,0
4997,4998,8331,934,Fridge,Electronics,5,398.66,2024-08-20 00:38:00,16,"New York, NY",...,204,1993.30,110,2024,8,3,53.92,-22,1,0
4998,4999,7505,439,Laptop,Appliances,3,1000.95,2024-08-26 11:05:00,16,"Miami, FL",...,144,3002.85,344,2024,8,3,238.89,-69,1,0


## Promotion Effectiveness Analysis 
***Business Problem :
Walmart gives promotions, but management wants to know whether promotions actually improve sales or create inventory pressure.***

In [4]:
# What percentage of transactions had promotions? 
query = """
select 
    round(
        count(distinct case when promotion_applied = 1 then transaction_id end) 
        / count(distinct transaction_id) * 100,
        2
    ) as promotion_transaction_percentage
from walmart_sales;
"""

result = pd.read_sql(query, engine)
result

,promotion_transaction_percentage
0,52.14


In [5]:
# Which promotion type generated the highest revenue?

query = """
select 
    promotion_type,
    round(sum(quantity_sold * unit_price), 2) as total_revenue
from walmart_sales
where promotion_applied = 1
group by promotion_type
order by total_revenue desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,promotion_type,total_revenue
0,No Promotion,5439571.31


In [6]:
# Do promoted products sell more quantity?
query = """
select 
    case
        when promotion_applied = 1 then 'promoted'
        else 'not promoted'
    end as promotion_status,
    sum(quantity_sold) as total_quantity_sold,
    round(avg(quantity_sold), 2) as avg_quantity_sold
from walmart_sales
group by promotion_applied;
"""

result = pd.read_sql(query, engine)
result

,promotion_status,total_quantity_sold,avg_quantity_sold
0,promoted,7780.0,2.98
1,not promoted,7134.0,2.98


In [7]:
# Do promoted products have higher actual demand?
query = """
select 
    case
        when promotion_applied = 1 then 'promoted'
        else 'not promoted'
    end as promotion_status,
    sum(actual_demand) as total_actual_demand,
    round(avg(actual_demand), 2) as avg_actual_demand
from walmart_sales
group by promotion_applied;
"""

result = pd.read_sql(query, engine)
result

,promotion_status,total_actual_demand,avg_actual_demand
0,promoted,776747.0,297.95
1,not promoted,718695.0,300.33


In [8]:
# Do promotions increase stockout risk?
query = """
select 
    case
        when promotion_applied = 1 then 'promoted'
        else 'not promoted'
    end as promotion_status,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by promotion_applied;
"""

result = pd.read_sql(query, engine)
result

,promotion_status,stockout_rate_percentage
0,promoted,52.51
1,not promoted,51.15


In [9]:
# Which category benefits most from promotions?
query = """
select 
    category,
    round(sum(quantity_sold * unit_price), 2) as promotion_revenue,
    sum(quantity_sold) as promotion_quantity_sold,
    round(avg(actual_demand), 2) as avg_actual_demand
from walmart_sales
where promotion_applied = 1
group by category
order by promotion_revenue desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,category,promotion_revenue,promotion_quantity_sold,avg_actual_demand
0,Electronics,4125922.81,3993.0,302.31


In [10]:
# Which store uses promotions most?
query = """
select 
    store_id,
    store_location,
    round(
        sum(case when promotion_applied = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as promotion_usage_percentage
from walmart_sales
group by store_id, store_location
order by promotion_usage_percentage desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,store_id,store_location,promotion_usage_percentage
0,20,"Los Angeles, CA",68.75


In [11]:
# Which promotion type performs better: BOGO or Percentage Discount?
query = """
select distinct promotion_type
from walmart_sales;
"""

result = pd.read_sql(query, engine)
result

,promotion_type
0,No Promotion
1,Percentage Discount
2,
3,BOGO
4,


In [12]:
# business question: which promotion type performs better: bogo or percentage discount?

query = """
select 
    promotion_type,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    sum(quantity_sold) as total_quantity_sold,
    round(avg(actual_demand), 2) as avg_actual_demand,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
where lower(promotion_type) in ('bogo', 'percentage discount')
group by promotion_type
order by total_revenue desc;
"""

result = pd.read_sql(query, engine)
result

,promotion_type,total_revenue,total_quantity_sold,avg_actual_demand,stockout_rate_percentage
0,BOGO,2534988.18,2465.0,292.36,52.07
1,Percentage Discount,2400750.12,2327.0,296.21,50.97


In [13]:
# Are promotions causing demand spikes?

query = """
select 
    case
        when promotion_applied = 1 then 'promoted'
        else 'not promoted'
    end as promotion_status,
    round(avg(actual_demand), 2) as avg_actual_demand,
    max(actual_demand) as max_actual_demand,
    round(avg(forecasted_demand), 2) as avg_forecasted_demand
from walmart_sales
group by promotion_applied;
"""

result = pd.read_sql(query, engine)
result


,promotion_status,avg_actual_demand,max_actual_demand,avg_forecasted_demand
0,promoted,297.95,510,299.85
1,not promoted,300.33,510,294.17


In [14]:
# business question: how do promoted and non-promoted products compare on revenue, quantity sold, demand, and stockout risk?

query = """
select
    case
        when promotion_applied = 1 then 'promoted'
        else 'not promoted'
    end as promotion_status,

    round(sum(quantity_sold * unit_price), 2) as revenue,

    round(avg(quantity_sold), 2) as avg_quantity_sold,

    round(avg(actual_demand), 2) as avg_demand,

    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage

from walmart_sales
group by promotion_applied
order by revenue desc;
"""

result = pd.read_sql(query, engine)
result

,promotion_status,revenue,avg_quantity_sold,avg_demand,stockout_rate_percentage
0,promoted,8062411.03,2.98,297.95,52.51
1,not promoted,7201190.42,2.98,300.33,51.15


## Business insight

***Promoted products generated higher/lower revenue compared to non-promoted products. 
If promoted products also have higher stockout rate, promotions are increasing demand but creating inventory pressure.
Walmart should continue promotions only where revenue uplift is strong and stockout risk is controlled.***

## Customer Loyalty Analysis 

***business question: which customer segments generate the highest value, respond best to promotions, and should be targeted for future marketing campaigns?***

***purpose: analyze customer loyalty, age, gender, income, payment behavior, and promotion response to identify the most valuable customer segments for targeted campaigns.***

In [15]:
query = """
with base_data as (
    select
        transaction_id,
        customer_id,
        customer_loyalty_level,
        customer_age,
        customer_gender,
        customer_income,
        payment_method,
        promotion_applied,
        quantity_sold,
        unit_price,

        case
            when customer_age < 25 then 'under 25'
            when customer_age between 25 and 34 then '25-34'
            when customer_age between 35 and 44 then '35-44'
            when customer_age between 45 and 54 then '45-54'
            else '55+'
        end as age_group,

        case
            when customer_income < 30000 then 'low income'
            when customer_income between 30000 and 70000 then 'middle income'
            else 'high income'
        end as income_group

    from walmart_sales
),

loyalty_revenue as (
    select 
        customer_loyalty_level,
        round(sum(quantity_sold * unit_price), 2) as total_revenue
    from base_data
    group by customer_loyalty_level
    order by total_revenue desc
    limit 1
),

loyalty_atv as (
    select 
        customer_loyalty_level,
        round(sum(quantity_sold * unit_price) / nullif(count(distinct transaction_id), 0), 2) as avg_transaction_value
    from base_data
    group by customer_loyalty_level
    order by avg_transaction_value desc
    limit 1
),

loyalty_quantity as (
    select 
        customer_loyalty_level,
        sum(quantity_sold) as total_quantity_sold
    from base_data
    group by customer_loyalty_level
    order by total_quantity_sold desc
    limit 1
),

age_revenue as (
    select 
        age_group,
        round(sum(quantity_sold * unit_price), 2) as total_revenue
    from base_data
    group by age_group
    order by total_revenue desc
    limit 1
),

gender_revenue as (
    select 
        customer_gender,
        round(sum(quantity_sold * unit_price), 2) as total_revenue
    from base_data
    group by customer_gender
    order by total_revenue desc
    limit 1
),

income_spend as (
    select 
        income_group,
        round(sum(quantity_sold * unit_price), 2) as total_revenue
    from base_data
    group by income_group
    order by total_revenue desc
    limit 1
),

customer_spending as (
    select 
        customer_id,
        sum(quantity_sold * unit_price) as customer_total_spend
    from base_data
    group by customer_id
),

average_spending as (
    select avg(customer_total_spend) as avg_customer_spend
    from customer_spending
),

high_value_customers as (
    select cs.customer_id
    from customer_spending cs
    cross join average_spending av
    where cs.customer_total_spend > av.avg_customer_spend
),

high_value_payment as (
    select 
        b.payment_method,
        count(*) as usage_count,
        round(sum(b.quantity_sold * b.unit_price), 2) as total_revenue
    from base_data b
    join high_value_customers h
        on b.customer_id = h.customer_id
    group by b.payment_method
    order by usage_count desc, total_revenue desc
    limit 1
),

platinum_bronze as (
    select 
        customer_loyalty_level,
        round(sum(quantity_sold * unit_price) / nullif(count(distinct customer_id), 0), 2) as revenue_per_customer
    from base_data
    where lower(customer_loyalty_level) in ('platinum', 'bronze')
    group by customer_loyalty_level
),

platinum_vs_bronze as (
    select
        case
            when 
                max(case when lower(customer_loyalty_level) = 'platinum' then revenue_per_customer end)
                >
                max(case when lower(customer_loyalty_level) = 'bronze' then revenue_per_customer end)
            then 'platinum customers are more valuable than bronze customers'
            else 'bronze customers are more valuable than or equal to platinum customers'
        end as comparison_result
    from platinum_bronze
),

target_segment as (
    select 
        customer_loyalty_level,
        age_group,
        round(sum(quantity_sold * unit_price), 2) as total_revenue,
        round(
            sum(case when promotion_applied = 1 then 1 else 0 end) / count(*) * 100.0,
            2
        ) as promotion_usage_percentage
    from base_data
    group by customer_loyalty_level, age_group
    order by total_revenue desc, promotion_usage_percentage desc
    limit 1
),

loyalty_promotion as (
    select 
        customer_loyalty_level,
        round(
            sum(case when promotion_applied = 1 then 1 else 0 end) / count(*) * 100.0,
            2
        ) as promotion_usage_percentage
    from base_data
    group by customer_loyalty_level
    order by promotion_usage_percentage desc
    limit 1
)

select
    lr.customer_loyalty_level as highest_revenue_loyalty_level,
    lr.total_revenue as highest_loyalty_revenue,

    la.customer_loyalty_level as highest_atv_loyalty_level,
    la.avg_transaction_value,

    lq.customer_loyalty_level as highest_quantity_loyalty_level,
    lq.total_quantity_sold,

    ar.age_group as highest_revenue_age_group,
    ar.total_revenue as age_group_revenue,

    gr.customer_gender as highest_revenue_gender_group,
    gr.total_revenue as gender_group_revenue,

    inc.income_group as highest_spending_income_group,
    inc.total_revenue as income_group_revenue,

    hvp.payment_method as preferred_payment_method_by_high_value_customers,

    pvb.comparison_result as platinum_vs_bronze_result,

    concat(ts.customer_loyalty_level, ' customers in ', ts.age_group, ' segment') as recommended_campaign_segment,
    ts.total_revenue as campaign_segment_revenue,
    ts.promotion_usage_percentage as campaign_segment_promotion_usage,

    lp.customer_loyalty_level as highest_promotion_usage_loyalty_level,
    lp.promotion_usage_percentage as loyalty_promotion_usage_percentage

from loyalty_revenue lr
cross join loyalty_atv la
cross join loyalty_quantity lq
cross join age_revenue ar
cross join gender_revenue gr
cross join income_spend inc
cross join high_value_payment hvp
cross join platinum_vs_bronze pvb
cross join target_segment ts
cross join loyalty_promotion lp;
"""

result = pd.read_sql(query, engine)
result

,highest_revenue_loyalty_level,highest_loyalty_revenue,highest_atv_loyalty_level,avg_transaction_value,highest_quantity_loyalty_level,total_quantity_sold,highest_revenue_age_group,age_group_revenue,highest_revenue_gender_group,gender_group_revenue,highest_spending_income_group,income_group_revenue,preferred_payment_method_by_high_value_customers,platinum_vs_bronze_result,recommended_campaign_segment,campaign_segment_revenue,campaign_segment_promotion_usage,highest_promotion_usage_loyalty_level,loyalty_promotion_usage_percentage
0,Platinum,4012963.14,Platinum,3089.27,Platinum,3896.0,55+,4648642.34,Female,5193570.83,high income,7677585.99,Digital Wallet,platinum customers are more valuable than bron...,Platinum customers in 55+ segment,1224868.76,51.23,Silver,52.8


In [16]:
# #Add new columns
query = """
alter table walmart_sales
add column age_group varchar(20),
add column income_group varchar(30);
"""

with engine.begin() as conn:
    conn.execute(text(query))

In [17]:
# Update age group and income group
query = """
update walmart_sales
set
    age_group = case
        when customer_age between 18 and 25 then 'young'
        when customer_age between 26 and 40 then 'adult'
        when customer_age between 41 and 60 then 'middle age'
        when customer_age > 60 then 'senior'
        else 'unknown'
    end,

    income_group = case
        when customer_income < 40000 then 'low income'
        when customer_income between 40000 and 70000 then 'middle income'
        when customer_income between 70001 and 100000 then 'high income'
        when customer_income > 100000 then 'premium income'
        else 'unknown'
    end;
"""

with engine.begin() as conn:
    conn.execute(text(query))

In [18]:
query = """
select
    customer_age,
    age_group,
    customer_income,
    income_group
from walmart_sales
limit 10;
"""

result = pd.read_sql(query, engine)
result

,customer_age,age_group,customer_income,income_group
0,29,adult,98760.83,high income
1,34,adult,69781.93,middle income
2,69,senior,77373.10,high income
3,47,middle age,33383.04,low income
4,70,senior,108999.41,premium income
5,40,adult,31779.77,low income
6,54,middle age,75813.17,high income
7,57,middle age,115784.29,premium income
8,64,senior,77392.78,high income
9,24,young,44787.70,middle income


In [19]:
# Analyze segment revenue

query = """
select
    age_group,
    income_group,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    sum(quantity_sold) as total_quantity_sold,
    count(distinct customer_id) as total_customers
from walmart_sales
group by age_group, income_group
order by total_revenue desc;
"""

result = pd.read_sql(query, engine)
result

,age_group,income_group,total_revenue,total_quantity_sold,total_customers
0,middle age,high income,1811096.24,1786.0,549
1,middle age,middle income,1668755.78,1591.0,521
2,middle age,low income,1297174.97,1259.0,396
3,adult,middle income,1288261.71,1220.0,402
4,adult,high income,1288047.29,1271.0,408
5,middle age,premium income,1124860.18,1107.0,357
6,adult,premium income,985431.06,919.0,301
7,adult,low income,837230.51,806.0,269
8,senior,middle income,834723.16,843.0,285
9,senior,high income,790474.46,798.0,274


## Supplier & Reorder Analysis
***Business Problem :
Supplier delay can worsen stockout problems. Walmart needs to identify supplier-related inventory risks.***

In [20]:
# Which suppliers have the highest average lead time?
query = """
select 
supplier_id,
round(avg(supplier_lead_time),2) as avg_supplier_lead_time,
count(distinct product_id) as total_products
from walmart_sales
group by supplier_id
order by avg_supplier_lead_time desc
limit 10;
"""
result = pd.read_sql(query,engine)
result

,supplier_id,avg_supplier_lead_time,total_products
0,321,8.63,8
1,106,7.88,8
2,450,7.80,5
3,253,7.70,10
4,193,7.50,6
5,370,7.45,11
6,388,7.36,11
7,162,7.27,11
8,411,7.27,11
9,207,7.25,16


In [21]:
# Which suppliers are linked to most stockout cases?
query = """
select
    supplier_id,
    sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_cases,
    round(sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100, 2) as stockout_rate
from walmart_sales
group by supplier_id
order by stockout_cases desc
limit 10;
"""

result = pd.read_sql(query, engine)
result

,supplier_id,stockout_cases,stockout_rate
0,470,16.0,66.67
1,442,14.0,77.78
2,289,14.0,70.00
3,405,14.0,73.68
4,333,13.0,100.00
5,283,13.0,81.25
6,366,13.0,68.42
7,170,13.0,86.67
8,231,13.0,76.47
9,131,13.0,65.00


In [22]:
# Which suppliers provide products below reorder point?
query = """
select
    supplier_id,
    count(*) as below_reorder_records,
    count(distinct product_id) as affected_products
from walmart_sales
where inventory_level <= reorder_point
group by supplier_id
order by below_reorder_records desc
limit 10;
"""

result = pd.read_sql(query, engine)
result

,supplier_id,below_reorder_records,affected_products
0,238,6,6
1,427,6,6
2,457,6,6
3,107,6,6
4,186,6,6
5,156,6,6
6,442,6,6
7,153,6,6
8,191,5,5
9,190,5,5


In [23]:
# Which stores depend on slow suppliers?
query = """
with avg_lead_time as (
    select avg(supplier_lead_time) as overall_avg_lead_time
    from walmart_sales
)
select
    store_id,
    store_location,
    count(distinct supplier_id) as slow_supplier_count,
    round(avg(supplier_lead_time), 2) as avg_store_supplier_lead_time
from walmart_sales
cross join avg_lead_time
where supplier_lead_time > overall_avg_lead_time
group by store_id, store_location
order by slow_supplier_count desc, avg_store_supplier_lead_time desc
limit 10;
"""

result = pd.read_sql(query, engine)
result

,store_id,store_location,slow_supplier_count,avg_store_supplier_lead_time
0,14,"New York, NY",39,8.00
1,3,"Los Angeles, CA",38,8.13
2,5,"Miami, FL",36,7.76
3,4,"Los Angeles, CA",35,8.14
4,8,"New York, NY",35,8.03
5,2,"Chicago, IL",34,7.80
6,17,"Dallas, TX",33,8.58
7,20,"New York, NY",33,7.94
8,11,"New York, NY",33,7.82
9,11,"Dallas, TX",32,8.27


In [24]:
# Which products have high reorder quantity?
query = """
select
    product_id,
    product_name,
    category,
    sum(reorder_quantity) as total_reorder_quantity,
    round(avg(reorder_quantity), 2) as avg_reorder_quantity
from walmart_sales
group by product_id, product_name, category
order by total_reorder_quantity desc
limit 10;
"""

result = pd.read_sql(query, engine)
result

,product_id,product_name,category,total_reorder_quantity,avg_reorder_quantity
0,155,Fridge,Appliances,1182.0,236.40
1,776,Smartphone,Appliances,1022.0,255.50
2,178,Washing Machine,Electronics,980.0,245.00
3,465,Headphones,Electronics,845.0,211.25
4,298,Camera,Appliances,830.0,276.67
5,603,Headphones,Appliances,753.0,251.00
6,954,Tablet,Appliances,742.0,247.33
7,142,Fridge,Electronics,734.0,244.67
8,343,Smartphone,Electronics,728.0,242.67
9,260,Smartphone,Appliances,726.0,242.00


In [25]:
# Which products need urgent replenishment?
query = """
select
    product_id,
    product_name,
    category,
    supplier_id,
    inventory_level,
    reorder_point,
    actual_demand,
    reorder_quantity,
    supplier_lead_time
from walmart_sales
where inventory_level <= reorder_point
   or inventory_level < actual_demand
order by supplier_lead_time desc, actual_demand desc
limit 20;
"""

result = pd.read_sql(query, engine)
result

,product_id,product_name,category,supplier_id,inventory_level,reorder_point,actual_demand,reorder_quantity,supplier_lead_time
0,287,Laptop,Appliances,139,165,76,510,180,10
1,393,Headphones,Appliances,206,369,69,510,102,10
2,199,Headphones,Appliances,412,281,59,510,222,10
3,233,TV,Appliances,415,498,76,509,192,10
4,660,TV,Appliances,331,232,68,507,103,10
5,943,Headphones,Appliances,168,15,87,507,231,10
6,573,Headphones,Electronics,271,324,50,506,250,10
7,255,Tablet,Appliances,303,237,101,505,236,10
8,110,Washing Machine,Electronics,206,246,114,504,257,10
9,834,Washing Machine,Electronics,175,372,106,504,295,10


In [26]:
# Which suppliers should be reviewed?
query = """
select
    supplier_id,
    round(avg(supplier_lead_time), 2) as avg_lead_time,
    sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_cases,
    round(sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100, 2) as stockout_rate,
    sum(case when inventory_level <= reorder_point then 1 else 0 end) as below_reorder_cases
from walmart_sales
group by supplier_id
order by stockout_rate desc, avg_lead_time desc, below_reorder_cases desc
limit 10;
"""

result = pd.read_sql(query, engine)
result

,supplier_id,avg_lead_time,stockout_cases,stockout_rate,below_reorder_cases
0,333,6.69,13.0,100.00,3.0
1,146,6.50,4.0,100.00,0.0
2,104,6.22,8.0,88.89,0.0
3,115,5.11,8.0,88.89,1.0
4,305,5.88,7.0,87.50,1.0
5,170,6.07,13.0,86.67,3.0
6,156,5.31,11.0,84.62,6.0
7,474,4.08,10.0,83.33,1.0
8,275,6.55,9.0,81.82,3.0
9,425,5.91,9.0,81.82,2.0


In [27]:
# Is higher supplier lead time related to stockout?
query = """
select
    case
        when supplier_lead_time <= 3 then 'fast supplier'
        when supplier_lead_time between 4 and 7 then 'medium supplier'
        else 'slow supplier'
    end as lead_time_group,
    count(*) as total_records,
    round(avg(supplier_lead_time), 2) as avg_lead_time,
    round(sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100, 2) as stockout_rate
from walmart_sales
group by lead_time_group
order by avg_lead_time;
"""

result = pd.read_sql(query, engine)
result

,lead_time_group,total_records,avg_lead_time,stockout_rate
0,fast supplier,1483,2.01,51.72
1,medium supplier,2020,5.52,51.09
2,slow supplier,1497,9.00,53.04


In [28]:
# Which category has the longest supplier lead time?
query = """
select
    category,
    round(avg(supplier_lead_time), 2) as avg_supplier_lead_time
from walmart_sales
group by category
order by avg_supplier_lead_time desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,category,avg_supplier_lead_time
0,Electronics,5.59


In [29]:
# Which supplier-product combinations are risky?

query = """
select
    supplier_id,
    product_id,
    product_name,
    category,
    round(avg(supplier_lead_time), 2) as avg_lead_time,
    sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_cases,
    sum(case when inventory_level <= reorder_point then 1 else 0 end) as below_reorder_cases,
    round(avg(actual_demand), 2) as avg_actual_demand
from walmart_sales
group by supplier_id, product_id, product_name, category
order by stockout_cases desc, below_reorder_cases desc, avg_lead_time desc
limit 20;
"""

result = pd.read_sql(query, engine)
result

,supplier_id,product_id,product_name,category,avg_lead_time,stockout_cases,below_reorder_cases,avg_actual_demand
0,344,883,Fridge,Appliances,3.0,2.0,0.0,133.0
1,228,953,Camera,Appliances,10.0,1.0,1.0,113.0
2,257,887,Headphones,Electronics,10.0,1.0,1.0,409.0
3,109,713,Smartphone,Electronics,10.0,1.0,1.0,499.0
4,267,576,Fridge,Electronics,10.0,1.0,1.0,203.0
5,206,277,TV,Electronics,10.0,1.0,1.0,164.0
6,477,659,Fridge,Appliances,10.0,1.0,1.0,130.0
7,314,636,Tablet,Electronics,10.0,1.0,1.0,481.0
8,366,759,Laptop,Appliances,10.0,1.0,1.0,146.0
9,169,847,Fridge,Electronics,10.0,1.0,1.0,162.0


In [30]:
query = """
with supplier_scorecard as (
    select
        supplier_id,
        round(avg(supplier_lead_time), 2) as avg_lead_time,
        count(distinct product_id) as total_products,
        count(distinct store_id) as total_stores,
        
        sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_cases,
        
        round(
            sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
            2
        ) as stockout_rate,
        
        sum(case when inventory_level <= reorder_point then 1 else 0 end) as below_reorder_cases,
        
        sum(case when inventory_level < actual_demand then 1 else 0 end) as understock_cases,
        
        round(avg(reorder_quantity), 2) as avg_reorder_quantity
    from walmart_sales
    group by supplier_id
)

select
    supplier_id,
    avg_lead_time,
    total_products,
    total_stores,
    stockout_cases,
    stockout_rate,
    below_reorder_cases,
    understock_cases,
    avg_reorder_quantity,

    round(
        stockout_rate + avg_lead_time + below_reorder_cases + understock_cases,
        2
    ) as supplier_risk_score,

    case
        when stockout_rate > 50 and avg_lead_time > 7 then
            concat('supplier ', supplier_id, ' should be reviewed urgently because stockout rate and lead time are high.')
        
        when below_reorder_cases > 0 and understock_cases > 0 then
            concat('supplier ', supplier_id, ' needs replenishment planning improvement because many products are below reorder point or understocked.')
        
        when avg_lead_time > 7 then
            concat('supplier ', supplier_id, ' has slow delivery performance and should be monitored.')
        
        else
            concat('supplier ', supplier_id, ' is currently stable based on supplier risk metrics.')
    end as business_recommendation

from supplier_scorecard
order by supplier_risk_score desc;
"""

supplier_scorecard = pd.read_sql(query, engine)
supplier_scorecard

,supplier_id,avg_lead_time,total_products,total_stores,stockout_cases,stockout_rate,below_reorder_cases,understock_cases,avg_reorder_quantity,supplier_risk_score,business_recommendation
0,333,6.69,13,8,13.0,100.00,3.0,8.0,215.85,117.69,supplier 333 needs replenishment planning impr...
1,146,6.50,4,4,4.0,100.00,0.0,1.0,176.25,107.50,supplier 146 is currently stable based on supp...
2,156,5.31,13,11,11.0,84.62,6.0,9.0,218.15,104.93,supplier 156 needs replenishment planning impr...
3,170,6.07,15,12,13.0,86.67,3.0,7.0,234.27,102.74,supplier 170 needs replenishment planning impr...
4,442,5.61,18,10,14.0,77.78,6.0,13.0,185.44,102.39,supplier 442 needs replenishment planning impr...
...,...,...,...,...,...,...,...,...,...,...,...
396,385,5.83,12,7,3.0,25.00,1.0,5.0,189.00,36.83,supplier 385 needs replenishment planning impr...
397,296,6.00,8,7,2.0,25.00,1.0,4.0,231.50,36.00,supplier 296 needs replenishment planning impr...
398,280,6.43,6,7,1.0,14.29,1.0,4.0,196.86,25.72,supplier 280 needs replenishment planning impr...
399,272,7.09,11,7,1.0,9.09,2.0,6.0,192.91,24.18,supplier 272 needs replenishment planning impr...


## Weather and Holiday Impact Analysis 
***Business Problem : 
Sales and demand may change during holidays or different weather conditions***

In [31]:
# Does revenue increase during holidays?
query = """
select
    case
        when holiday_indicator = 1 then 'holiday'
        else 'non-holiday'
    end as holiday_status,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    round(avg(quantity_sold * unit_price), 2) as avg_revenue_per_record
from walmart_sales
group by holiday_indicator
order by total_revenue desc;
"""

result = pd.read_sql(query, engine)
result

,holiday_status,total_revenue,avg_revenue_per_record
0,holiday,7668342.57,3068.56
1,non-holiday,7595258.88,3036.89


In [32]:
# Does actual demand increase during holidays?
query = """
select
    case
        when holiday_indicator = 1 then 'holiday'
        else 'non-holiday'
    end as holiday_status,
    sum(actual_demand) as total_actual_demand,
    round(avg(actual_demand), 2) as avg_actual_demand
from walmart_sales
group by holiday_indicator
order by avg_actual_demand desc;
"""

result = pd.read_sql(query, engine)
result

,holiday_status,total_actual_demand,avg_actual_demand
0,holiday,749003.0,299.72
1,non-holiday,746439.0,298.46


In [33]:
# Is stockout rate higher on holidays?
query = """
select
    case
        when holiday_indicator = 1 then 'holiday'
        else 'non-holiday'
    end as holiday_status,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by holiday_indicator
order by stockout_rate_percentage desc;
"""

result = pd.read_sql(query, engine)
result

,holiday_status,stockout_rate_percentage
0,holiday,52.14
1,non-holiday,51.58


In [34]:
# Which category sells more during holidays?
query = """
select
    category,
    sum(quantity_sold) as total_quantity_sold,
    round(sum(quantity_sold * unit_price), 2) as total_revenue
from walmart_sales
where holiday_indicator = 1
group by category
order by total_quantity_sold desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,category,total_quantity_sold,total_revenue
0,Electronics,3880.0,3999682.67


In [35]:
# Which weather condition has the highest revenue?
query = """
select
    weather_conditions,
    round(sum(quantity_sold * unit_price), 2) as total_revenue
from walmart_sales
group by weather_conditions
order by total_revenue desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,weather_conditions,total_revenue
0,Sunny,3979603.69


In [36]:
# Which weather condition has higher stockout rate?
query = """
select
    weather_conditions,
    count(*) as total_records,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by weather_conditions
order by stockout_rate_percentage desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,weather_conditions,total_records,stockout_rate_percentage
0,Rainy,1218,54.02


In [37]:
# Is forecast error higher during holidays?
query = """
select
    case
        when holiday_indicator = 1 then 'holiday'
        else 'non-holiday'
    end as holiday_status,
    round(
        avg(
            abs(forecasted_demand - actual_demand) / nullif(actual_demand, 0) * 100
        ),
        2
    ) as avg_forecast_error_percentage
from walmart_sales
group by holiday_indicator
order by avg_forecast_error_percentage desc;
"""

result = pd.read_sql(query, engine)
result

,holiday_status,avg_forecast_error_percentage
0,holiday,60.74
1,non-holiday,59.11


In [38]:
# Do promotions perform better during holidays?
query = """
select
    case
        when holiday_indicator = 1 then 'holiday'
        else 'non-holiday'
    end as holiday_status,

    case
        when promotion_applied = 1 then 'promoted'
        else 'not promoted'
    end as promotion_status,

    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    sum(quantity_sold) as total_quantity_sold,
    round(avg(actual_demand), 2) as avg_actual_demand,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by holiday_indicator, promotion_applied
order by holiday_status, total_revenue desc;
"""

result = pd.read_sql(query, engine)
result

,holiday_status,promotion_status,total_revenue,total_quantity_sold,avg_actual_demand,stockout_rate_percentage
0,holiday,promoted,4054029.99,3899.0,296.66,51.87
1,holiday,not promoted,3614312.58,3557.0,303.08,52.43
2,non-holiday,promoted,4008381.04,3881.0,299.24,53.15
3,non-holiday,not promoted,3586877.84,3577.0,297.61,49.88


In [39]:
# Which weekday has the highest sales?
query = """
select
    weekday,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    sum(quantity_sold) as total_quantity_sold
from walmart_sales
group by weekday
order by total_revenue desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,weekday,total_revenue,total_quantity_sold
0,Thursday,2436640.72,2330.0


In [40]:
# Which weekday has the highest stockout rate?
query = """
select
    weekday,
    count(*) as total_records,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by weekday
order by stockout_rate_percentage desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,weekday,total_records,stockout_rate_percentage
0,Saturday,710,54.23


## Best business insight 
***Revenue and demand are higher/lower during holidays. If stockout rate and forecast error also increase during holidays, Walmart should improve holiday inventory planning and demand forecasting. Promotions should be targeted on high-performing holiday categories instead of applying promotions broadly.***

## Final Business Recommendation Questions 

In [41]:
# Which 5 products need immediate reorder?
query = """
select
    product_id,
    product_name,
    category,
    supplier_id,
    sum(inventory_level) as total_inventory,
    sum(actual_demand) as total_actual_demand,
    sum(reorder_quantity) as total_reorder_quantity,
    sum(case when inventory_level <= reorder_point then 1 else 0 end) as below_reorder_cases,
    sum(case when inventory_level < actual_demand then 1 else 0 end) as understock_cases
from walmart_sales
group by product_id, product_name, category, supplier_id
order by below_reorder_cases desc, understock_cases desc, total_actual_demand desc
limit 5;
"""

result = pd.read_sql(query, engine)
result

,product_id,product_name,category,supplier_id,total_inventory,total_actual_demand,total_reorder_quantity,below_reorder_cases,understock_cases
0,275,Headphones,Appliances,488,116.0,509.0,147.0,1.0,1.0
1,973,Fridge,Electronics,496,16.0,509.0,126.0,1.0,1.0
2,368,Headphones,Appliances,471,42.0,509.0,256.0,1.0,1.0
3,673,Headphones,Electronics,421,28.0,509.0,147.0,1.0,1.0
4,578,Camera,Electronics,100,97.0,508.0,172.0,1.0,1.0


In [42]:
# Which store needs operational improvement?
query = """
select
    store_id,
    store_location,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    round(sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100, 2) as stockout_rate,
    round(sum(case when inventory_level < actual_demand then 1 else 0 end) / count(*) * 100, 2) as understock_rate,
    round(avg(abs(forecasted_demand - actual_demand) / nullif(actual_demand, 0) * 100), 2) as forecast_error_percentage
from walmart_sales
group by store_id, store_location
order by stockout_rate desc, understock_rate desc, forecast_error_percentage desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,store_id,store_location,total_revenue,stockout_rate,understock_rate,forecast_error_percentage
0,20,"New York, NY",195221.76,66.07,48.21,74.39


In [43]:
# Which category should Walmart focus on?
query = """
select
    category,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    sum(quantity_sold) as total_quantity_sold,
    sum(actual_demand) as total_actual_demand,
    round(sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100, 2) as stockout_rate
from walmart_sales
group by category
order by total_revenue desc, total_actual_demand desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,category,total_revenue,total_quantity_sold,total_actual_demand,stockout_rate
0,Electronics,7941631.8,7745.0,777940.0,52.88


In [44]:
# Which promotion type should be continued?
query = """
select
    promotion_type,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    sum(quantity_sold) as total_quantity_sold,
    round(avg(actual_demand), 2) as avg_actual_demand,
    round(sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100, 2) as stockout_rate
from walmart_sales
where promotion_applied = 1
group by promotion_type
order by total_revenue desc, total_quantity_sold desc, stockout_rate asc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,promotion_type,total_revenue,total_quantity_sold,avg_actual_demand,stockout_rate
0,No Promotion,5439571.31,5280.0,299.98,52.74


In [45]:
# Which promotion type should be reviewed?
query = """
select
    promotion_type,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    sum(quantity_sold) as total_quantity_sold,
    round(avg(actual_demand), 2) as avg_actual_demand,
    round(sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100, 2) as stockout_rate,
    round(avg(abs(forecasted_demand - actual_demand) / nullif(actual_demand, 0) * 100), 2) as forecast_error_percentage
from walmart_sales
where promotion_applied = 1
group by promotion_type
order by stockout_rate desc, forecast_error_percentage desc, total_revenue asc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,promotion_type,total_revenue,total_quantity_sold,avg_actual_demand,stockout_rate,forecast_error_percentage
0,,12423.24,12.0,283.33,66.67,19.38


In [46]:
# Which suppliers need attention?
query = """
select
    supplier_id,
    round(avg(supplier_lead_time), 2) as avg_supplier_lead_time,
    count(distinct product_id) as total_products,
    sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_cases,
    sum(case when inventory_level <= reorder_point then 1 else 0 end) as below_reorder_cases,
    sum(case when inventory_level < actual_demand then 1 else 0 end) as understock_cases
from walmart_sales
group by supplier_id
order by stockout_cases desc, avg_supplier_lead_time desc, below_reorder_cases desc
limit 5;
"""

result = pd.read_sql(query, engine)
result

,supplier_id,avg_supplier_lead_time,total_products,stockout_cases,below_reorder_cases,understock_cases
0,470,5.79,24,16.0,2.0,15.0
1,442,5.61,18,14.0,6.0,13.0
2,289,5.60,20,14.0,3.0,11.0
3,405,5.32,18,14.0,0.0,9.0
4,333,6.69,13,13.0,3.0,8.0


In [47]:
# Where is forecasting failing?
query = """
select
    store_id,
    store_location,
    category,
    round(avg(abs(forecasted_demand - actual_demand) / nullif(actual_demand, 0) * 100), 2) as forecast_error_percentage,
    sum(actual_demand) as total_actual_demand,
    sum(forecasted_demand) as total_forecasted_demand
from walmart_sales
group by store_id, store_location, category
order by forecast_error_percentage desc
limit 10;
"""

result = pd.read_sql(query, engine)
result

,store_id,store_location,category,forecast_error_percentage,total_actual_demand,total_forecasted_demand
0,12,"New York, NY",Electronics,118.41,4552.0,7055.0
1,1,"Dallas, TX",Appliances,102.20,4839.0,5648.0
2,8,"Los Angeles, CA",Appliances,100.70,5379.0,7226.0
3,11,"Los Angeles, CA",Appliances,96.88,4068.0,4268.0
4,1,"Miami, FL",Appliances,89.78,7371.0,9477.0
5,13,"Los Angeles, CA",Electronics,85.27,8381.0,9137.0
6,6,"Chicago, IL",Electronics,84.75,6998.0,8159.0
7,12,"Miami, FL",Appliances,84.61,7209.0,6714.0
8,20,"New York, NY",Appliances,83.58,7785.0,8951.0
9,11,"Chicago, IL",Electronics,82.39,8200.0,9171.0


In [48]:
# Which customer group should be targeted?
query = """
select
    age_group,
    income_group,
    customer_loyalty_level,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    sum(quantity_sold) as total_quantity_sold,
    count(distinct customer_id) as total_customers,
    round(sum(case when promotion_applied = 1 then 1 else 0 end) / count(*) * 100, 2) as promotion_usage_rate
from walmart_sales
group by age_group, income_group, customer_loyalty_level
order by total_revenue desc, promotion_usage_rate desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,age_group,income_group,customer_loyalty_level,total_revenue,total_quantity_sold,total_customers,promotion_usage_rate
0,middle age,high income,Platinum,486797.86,469.0,144,47.26


In [49]:
# Which store has high revenue but high stockout risk?
query = """
with store_summary as (
    select
        store_id,
        store_location,
        round(sum(quantity_sold * unit_price), 2) as total_revenue,
        round(sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100, 2) as stockout_rate
    from walmart_sales
    group by store_id, store_location
),
store_avg as (
    select
        avg(total_revenue) as avg_revenue,
        avg(stockout_rate) as avg_stockout_rate
    from store_summary
)
select
    s.store_id,
    s.store_location,
    s.total_revenue,
    s.stockout_rate
from store_summary s
cross join store_avg a
where s.total_revenue > a.avg_revenue
and s.stockout_rate > a.avg_stockout_rate
order by s.stockout_rate desc, s.total_revenue desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,store_id,store_location,total_revenue,stockout_rate
0,20,"New York, NY",195221.76,66.07


In [50]:
#What are the top 5 actions management should take?
query = """
with product_risk as (
    select
        'reorder urgent products' as action_area,
        concat('reorder top understocked products because inventory is below demand or reorder point.') as management_action,
        sum(case when inventory_level <= reorder_point or inventory_level < actual_demand then 1 else 0 end) as risk_score
    from walmart_sales
),

store_risk as (
    select
        'improve store operations' as action_area,
        concat('focus on stores with high stockout rate, understock rate, and forecast error.') as management_action,
        sum(case when stockout_indicator = 1 then 1 else 0 end) as risk_score
    from walmart_sales
),

supplier_risk as (
    select
        'review suppliers' as action_area,
        concat('review suppliers with high lead time, stockout cases, and below-reorder products.') as management_action,
        round(avg(supplier_lead_time), 2) + sum(case when stockout_indicator = 1 then 1 else 0 end) as risk_score
    from walmart_sales
),

forecast_risk as (
    select
        'improve forecasting' as action_area,
        concat('improve forecasting for stores and categories with high forecast error.') as management_action,
        round(avg(abs(forecasted_demand - actual_demand) / nullif(actual_demand, 0) * 100), 2) as risk_score
    from walmart_sales
),

promotion_risk as (
    select
        'optimize promotions' as action_area,
        concat('continue high-performing promotions and review promotions causing stockouts or forecast errors.') as management_action,
        sum(case when promotion_applied = 1 and stockout_indicator = 1 then 1 else 0 end) as risk_score
    from walmart_sales
)

select *
from product_risk

union all

select *
from store_risk

union all

select *
from supplier_risk

union all

select *
from forecast_risk

union all

select *
from promotion_risk

order by risk_score desc
limit 5;
"""

result = pd.read_sql(query, engine)
result

,action_area,management_action,risk_score
0,reorder urgent products,reorder top understocked products because inve...,2997.00
1,review suppliers,"review suppliers with high lead time, stockout...",2598.52
2,improve store operations,"focus on stores with high stockout rate, under...",2593.00
3,optimize promotions,continue high-performing promotions and review...,1369.00
4,improve forecasting,improve forecasting for stores and categories ...,59.93


***recommendation: walmart should reorder the top 5 understocked products because inventory is below demand and reorder risk is high.***

In [51]:
# Rank top 5 products by revenue in each category
query = """
with product_revenue as (
    select
        category,
        product_id,
        product_name,
        round(sum(quantity_sold * unit_price), 2) as total_revenue,
        row_number() over (
            partition by category
            order by sum(quantity_sold * unit_price) desc
        ) as revenue_rank
    from walmart_sales
    group by category, product_id, product_name
)
select
    category,
    product_id,
    product_name,
    total_revenue,
    revenue_rank
from product_revenue
where revenue_rank <= 5
order by category, revenue_rank;
"""

result = pd.read_sql(query, engine)
result

,category,product_id,product_name,total_revenue,revenue_rank
0,Appliances,155,Fridge,21093.16,1
1,Appliances,672,Laptop,18864.00,2
2,Appliances,800,Laptop,18744.53,3
3,Appliances,580,Fridge,17321.27,4
4,Appliances,154,Laptop,16406.90,5
5,Electronics,652,Camera,18813.36,1
6,Electronics,584,TV,18119.33,2
7,Electronics,465,Headphones,16291.22,3
8,Electronics,645,Washing Machine,16060.94,4
9,Electronics,523,Fridge,16032.00,5


In [52]:
# Find store-wise stockout rate
query = """
select
    store_id,
    store_location,
    count(*) as total_records,
    sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_cases,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by store_id, store_location
order by stockout_rate_percentage desc;
"""

result = pd.read_sql(query, engine)
result

,store_id,store_location,total_records,stockout_cases,stockout_rate_percentage
0,20,"New York, NY",56,37.0,66.07
1,5,"New York, NY",47,31.0,65.96
2,14,"New York, NY",73,48.0,65.75
3,16,"Los Angeles, CA",62,40.0,64.52
4,13,"Dallas, TX",58,37.0,63.79
...,...,...,...,...,...
95,20,"Chicago, IL",48,19.0,39.58
96,6,"Dallas, TX",48,18.0,37.50
97,5,"Miami, FL",59,22.0,37.29
98,14,"Chicago, IL",48,17.0,35.42


In [53]:
# Calculate monthly revenue growth
query = """
with monthly_revenue as (
    select
        date_format(transaction_date, '%%Y-%%m') as sales_month,
        sum(revenue) as total_revenue
    from walmart_sales
    group by date_format(transaction_date, '%%Y-%%m')
)
select
    sales_month,
    total_revenue,
    lag(total_revenue) over (order by sales_month) as prev_month_revenue,
    round(
        (total_revenue - lag(total_revenue) over (order by sales_month)) 
        / nullif(lag(total_revenue) over (order by sales_month), 0) * 100, 2
    ) as growth_percentage
from monthly_revenue
order by sales_month;
"""
result = pd.read_sql(query, engine)
result

,sales_month,total_revenue,prev_month_revenue,growth_percentage
0,2024-01,1731651.65,NaN,NaN
1,2024-02,1767062.38,1731651.65,2.04
2,2024-03,1833450.84,1767062.38,3.76
3,2024-04,1739800.75,1833450.84,-5.11
4,2024-05,1786559.47,1739800.75,2.69
5,2024-06,1600978.97,1786559.47,-10.39
6,2024-07,1878513.51,1600978.97,17.34
7,2024-08,2018315.81,1878513.51,7.44
8,2024-09,907268.07,2018315.81,-55.05


In [54]:
# Identify products below reorder point
query = """
select
    product_id,
    product_name,
    category,
    store_id,
    store_location,
    inventory_level,
    reorder_point,
    reorder_quantity
from walmart_sales
where inventory_level <= reorder_point
order by inventory_level asc, reorder_point desc;
"""

result = pd.read_sql(query, engine)
result

,product_id,product_name,category,store_id,store_location,inventory_level,reorder_point,reorder_quantity
0,356,Smartphone,Electronics,7,"Los Angeles, CA",0,148,276
1,407,Smartphone,Appliances,19,"Chicago, IL",0,130,151
2,974,Laptop,Electronics,10,"New York, NY",0,126,266
3,801,TV,Appliances,20,"Dallas, TX",0,126,145
4,765,Camera,Electronics,2,"Miami, FL",0,122,199
...,...,...,...,...,...,...,...,...
940,116,Headphones,Electronics,8,"Miami, FL",140,145,140
941,786,Tablet,Electronics,5,"Chicago, IL",141,142,265
942,431,Washing Machine,Appliances,2,"New York, NY",142,149,156
943,569,TV,Electronics,15,"Miami, FL",143,150,208


In [55]:
# Compare promotion vs non-promotion sales
query = """
select
    case
        when promotion_applied = 1 then 'promotion'
        else 'no promotion'
    end as promotion_status,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    sum(quantity_sold) as total_quantity_sold,
    round(avg(quantity_sold), 2) as avg_quantity_sold,
    round(avg(actual_demand), 2) as avg_actual_demand,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by promotion_applied
order by total_revenue desc;
"""

result = pd.read_sql(query, engine)
result

,promotion_status,total_revenue,total_quantity_sold,avg_quantity_sold,avg_actual_demand,stockout_rate_percentage
0,promotion,8062411.03,7780.0,2.98,297.95,52.51
1,no promotion,7201190.42,7134.0,2.98,300.33,51.15


In [56]:
# Calculate forecast error by product
query = """
select
    product_id,
    product_name,
    category,
    round(avg(abs(forecasted_demand - actual_demand)), 2) as avg_absolute_forecast_error,
    round(
        avg(abs(forecasted_demand - actual_demand) / nullif(actual_demand, 0) * 100),
        2
    ) as avg_forecast_error_percentage,
    sum(actual_demand) as total_actual_demand,
    sum(forecasted_demand) as total_forecasted_demand
from walmart_sales
group by product_id, product_name, category
order by avg_forecast_error_percentage desc;
"""

result = pd.read_sql(query, engine)
result

,product_id,product_name,category,avg_absolute_forecast_error,avg_forecast_error_percentage,total_actual_demand,total_forecasted_demand
0,142,Laptop,Electronics,406.0,446.15,91.0,497.0
1,109,Fridge,Electronics,405.0,426.32,95.0,500.0
2,813,Washing Machine,Electronics,381.0,423.33,90.0,471.0
3,394,Tablet,Appliances,396.0,416.84,95.0,491.0
4,842,Camera,Electronics,398.0,410.31,97.0,495.0
...,...,...,...,...,...,...,...
4249,312,Fridge,Appliances,0.0,0.00,411.0,411.0
4250,413,TV,Electronics,0.0,0.00,159.0,159.0
4251,311,Camera,Electronics,0.0,0.00,406.0,406.0
4252,807,Fridge,Appliances,0.0,0.00,229.0,229.0


In [57]:
# Create a store performance scorecard
query = """
select
    store_id,
    store_location,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    count(distinct transaction_id) as total_transactions,
    round(
        sum(quantity_sold * unit_price) / nullif(count(distinct transaction_id), 0),
        2
    ) as avg_transaction_value,
    sum(quantity_sold) as total_quantity_sold,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage,
    round(
        avg(abs(forecasted_demand - actual_demand) / nullif(actual_demand, 0) * 100),
        2
    ) as avg_forecast_error_percentage,
    round(
        sum(case when promotion_applied = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as promotion_usage_percentage
from walmart_sales
group by store_id, store_location
order by total_revenue desc;
"""

result = pd.read_sql(query, engine)
result

,store_id,store_location,total_revenue,total_transactions,avg_transaction_value,total_quantity_sold,stockout_rate_percentage,avg_forecast_error_percentage,promotion_usage_percentage
0,17,"Dallas, TX",213954.02,61,3507.44,190.0,40.98,59.17,52.46
1,17,"Chicago, IL",211660.03,58,3649.31,185.0,50.00,61.59,63.79
2,1,"Los Angeles, CA",210976.79,65,3245.80,196.0,53.85,55.45,53.85
3,11,"Chicago, IL",205997.80,51,4039.17,171.0,49.02,73.19,43.14
4,5,"Chicago, IL",205576.05,59,3484.34,175.0,62.71,60.10,59.32
...,...,...,...,...,...,...,...,...,...
95,14,"Miami, FL",107133.24,44,2434.85,105.0,50.00,45.97,47.73
96,4,"Dallas, TX",107053.65,39,2744.97,113.0,43.59,55.59,41.03
97,4,"Miami, FL",105486.08,40,2637.15,108.0,55.00,63.75,60.00
98,17,"New York, NY",91760.41,32,2867.51,90.0,56.25,46.65,50.00


In [58]:
# Find suppliers with high lead time and high stockout count.
query = """
with supplier_summary as (
    select
        supplier_id,
        round(avg(supplier_lead_time), 2) as avg_supplier_lead_time,
        sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_cases,
        count(distinct product_id) as total_products
    from walmart_sales
    group by supplier_id
),
supplier_average as (
    select
        avg(avg_supplier_lead_time) as overall_avg_lead_time,
        avg(stockout_cases) as overall_avg_stockout_cases
    from supplier_summary
)
select
    s.supplier_id,
    s.avg_supplier_lead_time,
    s.stockout_cases,
    s.total_products
from supplier_summary s
cross join supplier_average a
where s.avg_supplier_lead_time > a.overall_avg_lead_time
and s.stockout_cases > a.overall_avg_stockout_cases
order by s.stockout_cases desc, s.avg_supplier_lead_time desc;
"""

result = pd.read_sql(query, engine)
result


,supplier_id,avg_supplier_lead_time,stockout_cases,total_products
0,470,5.79,16.0,24
1,442,5.61,14.0,18
2,289,5.60,14.0,20
3,333,6.69,13.0,13
4,170,6.07,13.0,15
...,...,...,...,...
86,473,5.77,7.0,13
87,475,5.69,7.0,13
88,133,5.69,7.0,16
89,420,5.67,7.0,15


In [59]:
# Find high-demand products with low inventory
query = """
with product_summary as (
    select
        product_id,
        product_name,
        category,
        round(avg(inventory_level), 2) as avg_inventory_level,
        round(avg(actual_demand), 2) as avg_actual_demand,
        sum(quantity_sold) as total_quantity_sold
    from walmart_sales
    group by product_id, product_name, category
)
select
    product_id,
    product_name,
    category,
    avg_inventory_level,
    avg_actual_demand,
    total_quantity_sold,
    round(avg_actual_demand - avg_inventory_level, 2) as demand_inventory_gap
from product_summary
where avg_actual_demand > avg_inventory_level
order by demand_inventory_gap desc
limit 20;
"""

result = pd.read_sql(query, engine)
result

,product_id,product_name,category,avg_inventory_level,avg_actual_demand,total_quantity_sold,demand_inventory_gap
0,342,Camera,Electronics,1.0,504.0,2.0,503.0
1,469,Laptop,Electronics,3.0,503.0,1.0,500.0
2,943,Headphones,Appliances,15.0,507.0,4.0,492.0
3,607,Fridge,Electronics,13.0,504.0,1.0,491.0
4,979,TV,Appliances,18.0,506.0,2.0,488.0
5,743,Headphones,Appliances,21.0,508.0,2.0,487.0
6,713,TV,Appliances,21.0,503.0,4.0,482.0
7,673,Headphones,Electronics,28.0,509.0,3.0,481.0
8,119,Laptop,Appliances,17.0,495.0,2.0,478.0
9,288,Laptop,Appliances,26.0,502.0,1.0,476.0


In [60]:
# Create a monthly KPI summary view
from sqlalchemy import text

query = """
create or replace view monthly_kpi_summary as
select
    date_format(transaction_date, '%Y-%m') as sales_month,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    count(distinct transaction_id) as total_transactions,
    sum(quantity_sold) as total_quantity_sold,
    round(
        sum(quantity_sold * unit_price) / nullif(count(distinct transaction_id), 0),
        2
    ) as avg_transaction_value,
    sum(actual_demand) as total_actual_demand,
    sum(forecasted_demand) as total_forecasted_demand,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage,
    round(
        avg(abs(forecasted_demand - actual_demand) / nullif(actual_demand, 0) * 100),
        2
    ) as avg_forecast_error_percentage,
    round(
        sum(case when promotion_applied = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as promotion_usage_percentage
from walmart_sales
group by date_format(transaction_date, '%Y-%m');
"""

with engine.begin() as conn:
    conn.execute(text(query))

In [61]:
query = """
select *
from monthly_kpi_summary
order by sales_month;
"""

result = pd.read_sql(query, engine)
result

,sales_month,total_revenue,total_transactions,total_quantity_sold,avg_transaction_value,total_actual_demand,total_forecasted_demand,stockout_rate_percentage,avg_forecast_error_percentage,promotion_usage_percentage
0,2024-01,1731651.65,563,1672.0,3075.76,166932.0,167788.0,50.98,61.28,53.29
1,2024-02,1767062.38,560,1674.0,3155.47,169799.0,171914.0,46.07,60.63,52.14
2,2024-03,1833450.84,620,1809.0,2957.18,183080.0,181400.0,54.03,60.90,51.77
3,2024-04,1739800.75,561,1668.0,3101.25,169430.0,165247.0,51.87,59.34,50.45
4,2024-05,1786559.47,596,1768.0,2997.58,176216.0,172057.0,54.87,59.44,52.18
5,2024-06,1600978.97,542,1613.0,2953.84,170451.0,160850.0,52.77,56.44,49.08
6,2024-07,1878513.51,606,1833.0,3099.86,178184.0,184297.0,52.97,60.39,54.46
7,2024-08,2018315.81,656,1968.0,3076.70,193476.0,196832.0,52.74,60.79,54.27
8,2024-09,907268.07,296,909.0,3065.09,87874.0,85285.0,47.97,59.57,50.00


In [62]:
# what is the total revenue and transaction count?
query = """
select
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    count(distinct transaction_id) as total_transactions
from walmart_sales;
"""

result = pd.read_sql(query, engine)
result

,total_revenue,total_transactions
0,15263601.45,5000


In [63]:
# which category generates the highest revenue?
query = """
select
    category,
    round(sum(quantity_sold * unit_price), 2) as total_revenue
from walmart_sales
group by category
order by total_revenue desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,category,total_revenue
0,Electronics,7941631.8


In [64]:
# which store location performs best and worst?
query = """
with store_performance as (
    select
        store_location,
        round(sum(quantity_sold * unit_price), 2) as total_revenue,
        count(distinct transaction_id) as total_transactions,
        sum(quantity_sold) as total_quantity_sold
    from walmart_sales
    group by store_location
),

ranked_stores as (
    select
        store_location,
        total_revenue,
        total_transactions,
        total_quantity_sold,
        rank() over(order by total_revenue desc) as best_rank,
        rank() over(order by total_revenue asc) as worst_rank
    from store_performance
)

select
    'best store' as store_performance_status,
    store_location,
    total_revenue,
    total_transactions,
    total_quantity_sold
from ranked_stores
where best_rank = 1

union all

select
    'worst store' as store_performance_status,
    store_location,
    total_revenue,
    total_transactions,
    total_quantity_sold
from ranked_stores
where worst_rank = 1;
"""

result = pd.read_sql(query, engine)
result

,store_performance_status,store_location,total_revenue,total_transactions,total_quantity_sold
0,best store,"Los Angeles, CA",3276299.63,1038,3168.0
1,worst store,"Dallas, TX",2903930.74,998,2892.0


In [65]:
# what is the monthly revenue trend?
query = """
select
    date_format(transaction_date, '%%Y-%%m') as sales_month,
    round(sum(quantity_sold * unit_price), 2) as monthly_revenue,
    count(distinct transaction_id) as total_transactions,
    sum(quantity_sold) as total_quantity_sold
from walmart_sales
group by date_format(transaction_date, '%%Y-%%m')
order by sales_month;
"""

result = pd.read_sql(query, engine)
result

,sales_month,monthly_revenue,total_transactions,total_quantity_sold
0,2024-01,1731651.65,563,1672.0
1,2024-02,1767062.38,560,1674.0
2,2024-03,1833450.84,620,1809.0
3,2024-04,1739800.75,561,1668.0
4,2024-05,1786559.47,596,1768.0
5,2024-06,1600978.97,542,1613.0
6,2024-07,1878513.51,606,1833.0
7,2024-08,2018315.81,656,1968.0
8,2024-09,907268.07,296,909.0


In [66]:
# which products generate the highest revenue?
query = """
select
    product_id,
    product_name,
    category,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    sum(quantity_sold) as total_quantity_sold,
    count(distinct transaction_id) as total_transactions
from walmart_sales
group by product_id, product_name, category
order by total_revenue desc
limit 10;
"""

result = pd.read_sql(query, engine)
result

,product_id,product_name,category,total_revenue,total_quantity_sold,total_transactions
0,155,Fridge,Appliances,21093.16,18.0,5
1,672,Laptop,Appliances,18864.00,10.0,2
2,652,Camera,Electronics,18813.36,11.0,3
3,800,Laptop,Appliances,18744.53,10.0,3
4,584,TV,Electronics,18119.33,13.0,3
5,580,Fridge,Appliances,17321.27,14.0,3
6,154,Laptop,Appliances,16406.90,9.0,2
7,133,Camera,Appliances,16299.42,10.0,3
8,465,Headphones,Electronics,16291.22,10.0,4
9,431,TV,Appliances,16146.02,10.0,3


In [67]:
# which products have high demand but low inventory?
query = """
select
    product_id,
    product_name,
    category,
    store_location,
    sum(actual_demand) as total_actual_demand,
    round(avg(inventory_level), 2) as avg_inventory_level,
    round(avg(reorder_point), 2) as avg_reorder_point,
    round(sum(actual_demand) - avg(inventory_level), 2) as demand_inventory_gap
from walmart_sales
group by product_id, product_name, category, store_location
having total_actual_demand > avg_inventory_level
   and avg_inventory_level <= avg_reorder_point
order by demand_inventory_gap desc
limit 10;
"""

result = pd.read_sql(query, engine)
result

,product_id,product_name,category,store_location,total_actual_demand,avg_inventory_level,avg_reorder_point,demand_inventory_gap
0,801,TV,Appliances,"Dallas, TX",760.0,26.0,99.0,734.0
1,439,Camera,Appliances,"New York, NY",737.0,21.5,105.5,715.5
2,210,Smartphone,Electronics,"Dallas, TX",754.0,61.5,118.0,692.5
3,952,Fridge,Appliances,"Los Angeles, CA",751.0,71.5,86.5,679.5
4,652,Laptop,Electronics,"New York, NY",655.0,59.0,81.5,596.0
5,563,Tablet,Electronics,"Chicago, IL",674.0,91.5,98.0,582.5
6,643,Laptop,Appliances,"Los Angeles, CA",543.0,12.5,114.5,530.5
7,569,TV,Electronics,"Miami, FL",598.0,72.5,123.5,525.5
8,342,Camera,Electronics,"New York, NY",504.0,1.0,77.0,503.0
9,469,Laptop,Electronics,"Los Angeles, CA",503.0,3.0,128.0,500.0


In [68]:
# what is the stockout rate by store?
query = """
select
    store_location,
    count(*) as total_records,
    sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_count,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by store_location
order by stockout_rate_percentage desc;
"""

result = pd.read_sql(query, engine)
result

,store_location,total_records,stockout_count,stockout_rate_percentage
0,"New York, NY",987,547.0,55.42
1,"Los Angeles, CA",1038,542.0,52.22
2,"Chicago, IL",1013,519.0,51.23
3,"Miami, FL",964,489.0,50.73
4,"Dallas, TX",998,496.0,49.70


In [69]:
# which products are below reorder point?
query = """
select
    product_id,
    product_name,
    category,
    store_location,
    inventory_level,
    reorder_point,
    reorder_quantity
from walmart_sales
where inventory_level < reorder_point
order by inventory_level asc;
"""

result = pd.read_sql(query, engine)
result

,product_id,product_name,category,store_location,inventory_level,reorder_point,reorder_quantity
0,150,Laptop,Electronics,"Dallas, TX",0,117,225
1,566,Tablet,Appliances,"Los Angeles, CA",0,56,167
2,765,Camera,Electronics,"Miami, FL",0,122,199
3,119,Smartphone,Appliances,"Los Angeles, CA",0,76,266
4,878,Camera,Electronics,"New York, NY",0,53,154
...,...,...,...,...,...,...,...
933,183,Laptop,Electronics,"Miami, FL",140,150,193
934,786,Tablet,Electronics,"Chicago, IL",141,142,265
935,431,Washing Machine,Appliances,"New York, NY",142,149,156
936,569,TV,Electronics,"Miami, FL",143,150,208


In [70]:
# which suppliers have high lead time and high stockout count?
query = """
select
    supplier_id,
    round(avg(supplier_lead_time), 2) as avg_supplier_lead_time,
    sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_count,
    count(*) as total_records,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by supplier_id
having avg_supplier_lead_time > (
    select avg(supplier_lead_time)
    from walmart_sales
)
order by stockout_count desc, avg_supplier_lead_time desc;
"""

result = pd.read_sql(query, engine)
result

,supplier_id,avg_supplier_lead_time,stockout_count,total_records,stockout_rate_percentage
0,470,5.79,16.0,24,66.67
1,442,5.61,14.0,18,77.78
2,289,5.60,14.0,20,70.00
3,333,6.69,13.0,13,100.00
4,170,6.07,13.0,15,86.67
...,...,...,...,...,...
201,223,5.70,2.0,10,20.00
202,302,5.60,2.0,10,20.00
203,272,7.09,1.0,11,9.09
204,280,6.43,1.0,7,14.29


In [71]:
# what is the average forecast error percentage?
query = """
select
    round(
        avg(abs(actual_demand - forecasted_demand) / nullif(actual_demand, 0) * 100),
        2
    ) as average_forecast_error_percentage
from walmart_sales;
"""

result = pd.read_sql(query, engine)
result

,average_forecast_error_percentage
0,59.93


In [72]:
# which products have the highest forecast error?
query = """
select
    product_id,
    product_name,
    category,
    round(avg(abs(actual_demand - forecasted_demand)), 2) as avg_forecast_error,
    round(
        avg(abs(actual_demand - forecasted_demand) / nullif(actual_demand, 0) * 100),
        2
    ) as avg_forecast_error_percentage
from walmart_sales
group by product_id, product_name, category
order by avg_forecast_error_percentage desc
limit 10;
"""

result = pd.read_sql(query, engine)
result

,product_id,product_name,category,avg_forecast_error,avg_forecast_error_percentage
0,142,Laptop,Electronics,406.0,446.15
1,109,Fridge,Electronics,405.0,426.32
2,813,Washing Machine,Electronics,381.0,423.33
3,394,Tablet,Appliances,396.0,416.84
4,842,Camera,Electronics,398.0,410.31
5,772,Smartphone,Electronics,361.0,401.11
6,800,Fridge,Appliances,391.0,398.98
7,612,Laptop,Appliances,392.0,392.00
8,198,Fridge,Appliances,376.0,391.67
9,738,Fridge,Appliances,368.0,375.51


In [73]:
# is walmart under-forecasting or over-forecasting demand?
query = """
select
    case
        when forecasted_demand < actual_demand then 'under forecasting'
        when forecasted_demand > actual_demand then 'over forecasting'
        else 'accurate forecasting'
    end as forecast_status,
    count(*) as total_records,
    round(avg(abs(actual_demand - forecasted_demand)), 2) as avg_forecast_error,
    round(
        avg(abs(actual_demand - forecasted_demand) / nullif(actual_demand, 0) * 100),
        2
    ) as avg_forecast_error_percentage
from walmart_sales
group by forecast_status
order by total_records desc;
"""

result = pd.read_sql(query, engine)
result

,forecast_status,total_records,avg_forecast_error,avg_forecast_error_percentage
0,under forecasting,2555,136.47,35.50
1,over forecasting,2436,139.12,85.77
2,accurate forecasting,9,0.00,0.00


In [74]:
# do promotions increase revenue?
query = """
select
    case
        when promotion_applied = 1 then 'promotion applied'
        else 'no promotion'
    end as promotion_status,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    count(distinct transaction_id) as total_transactions,
    round(sum(quantity_sold * unit_price) / nullif(count(distinct transaction_id), 0), 2) as avg_transaction_value,
    sum(quantity_sold) as total_quantity_sold
from walmart_sales
group by promotion_status
order by total_revenue desc;
"""

result = pd.read_sql(query, engine)
result

,promotion_status,total_revenue,total_transactions,avg_transaction_value,total_quantity_sold
0,promotion applied,8062411.03,2607,3092.60,7780.0
1,no promotion,7201190.42,2393,3009.27,7134.0


In [75]:
# do promotions increase stockout risk?
query = """
select
    case
        when promotion_applied = 1 then 'promotion applied'
        else 'no promotion'
    end as promotion_status,
    count(*) as total_records,
    sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_count,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by promotion_status
order by stockout_rate_percentage desc;
"""

result = pd.read_sql(query, engine)
result

,promotion_status,total_records,stockout_count,stockout_rate_percentage
0,promotion applied,2607,1369.0,52.51
1,no promotion,2393,1224.0,51.15


In [76]:
# which promotion type performs best?
query = """
select
    promotion_type,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    count(distinct transaction_id) as total_transactions,
    round(sum(quantity_sold * unit_price) / nullif(count(distinct transaction_id), 0), 2) as avg_transaction_value,
    sum(quantity_sold) as total_quantity_sold,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
where promotion_applied = 1
group by promotion_type
order by total_revenue desc, avg_transaction_value desc;
"""

result = pd.read_sql(query, engine)
result

,promotion_type,total_revenue,total_transactions,avg_transaction_value,total_quantity_sold,stockout_rate_percentage
0,No Promotion,5439571.31,1790,3038.87,5280.0,52.74
1,BOGO,1368661.84,424,3227.98,1309.0,52.12
2,Percentage Discount,1239197.32,389,3185.60,1175.0,51.93
3,,12423.24,3,4141.08,12.0,66.67
4,,2557.32,1,2557.32,4.0,0.00


In [77]:
# which loyalty level generates the most revenue?
query = """
select
    customer_loyalty_level,
    round(sum(quantity_sold * unit_price), 2) as total_revenue,
    count(distinct transaction_id) as total_transactions,
    sum(quantity_sold) as total_quantity_sold,
    round(sum(quantity_sold * unit_price) / nullif(count(distinct transaction_id), 0), 2) as avg_transaction_value
from walmart_sales
group by customer_loyalty_level
order by total_revenue desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,customer_loyalty_level,total_revenue,total_transactions,total_quantity_sold,avg_transaction_value
0,Platinum,4012963.14,1299,3896.0,3089.27


In [78]:
# which payment method is most used by customers?
query = """
select
    payment_method,
    count(*) as usage_count,
    round(count(*) / (select count(*) from walmart_sales) * 100, 2) as usage_percentage,
    round(sum(quantity_sold * unit_price), 2) as total_revenue
from walmart_sales
group by payment_method
order by usage_count desc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,payment_method,usage_count,usage_percentage,total_revenue
0,Credit Card,1281,25.62,3829054.57


In [79]:
# does holiday demand increase stockout risk?
query = """
select
    case
        when holiday_indicator = 1 then 'holiday'
        else 'non holiday'
    end as holiday_status,
    count(*) as total_records,
    sum(actual_demand) as total_actual_demand,
    round(avg(actual_demand), 2) as avg_actual_demand,
    sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_count,
    round(
        sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
        2
    ) as stockout_rate_percentage
from walmart_sales
group by holiday_status
order by stockout_rate_percentage desc;
"""

result = pd.read_sql(query, engine)
result

,holiday_status,total_records,total_actual_demand,avg_actual_demand,stockout_count,stockout_rate_percentage
0,holiday,2499,749003.0,299.72,1303.0,52.14
1,non holiday,2501,746439.0,298.46,1290.0,51.58


In [80]:
# which store needs immediate business attention?
query = """
with store_metrics as (
    select
        store_location,
        round(sum(quantity_sold * unit_price), 2) as total_revenue,
        count(distinct transaction_id) as total_transactions,
        sum(quantity_sold) as total_quantity_sold,
        sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_count,
        round(
            sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
            2
        ) as stockout_rate_percentage,
        round(
            avg(abs(actual_demand - forecasted_demand) / nullif(actual_demand, 0) * 100),
            2
        ) as avg_forecast_error_percentage,
        sum(case when inventory_level < reorder_point then 1 else 0 end) as below_reorder_count
    from walmart_sales
    group by store_location
),

store_ranking as (
    select
        store_location,
        total_revenue,
        total_transactions,
        total_quantity_sold,
        stockout_count,
        stockout_rate_percentage,
        avg_forecast_error_percentage,
        below_reorder_count,

        rank() over(order by total_revenue asc) as low_revenue_rank,
        rank() over(order by stockout_rate_percentage desc) as stockout_risk_rank,
        rank() over(order by avg_forecast_error_percentage desc) as forecast_error_rank,
        rank() over(order by below_reorder_count desc) as reorder_risk_rank
    from store_metrics
)

select
    store_location,
    total_revenue,
    total_transactions,
    total_quantity_sold,
    stockout_count,
    stockout_rate_percentage,
    avg_forecast_error_percentage,
    below_reorder_count,
    low_revenue_rank,
    stockout_risk_rank,
    forecast_error_rank,
    reorder_risk_rank,
    (
        low_revenue_rank
        + stockout_risk_rank
        + forecast_error_rank
        + reorder_risk_rank
    ) as business_attention_score
from store_ranking
order by business_attention_score asc
limit 1;
"""

result = pd.read_sql(query, engine)
result

,store_location,total_revenue,total_transactions,total_quantity_sold,stockout_count,stockout_rate_percentage,avg_forecast_error_percentage,below_reorder_count,low_revenue_rank,stockout_risk_rank,forecast_error_rank,reorder_risk_rank,business_attention_score
0,"Miami, FL",2962567.02,964,2834.0,489.0,50.73,61.55,188.0,2,4,1,3,10


In [81]:
# what are the top 5 business recommendations for walmart?
query = """
with category_revenue as (
    select
        category,
        round(sum(quantity_sold * unit_price), 2) as total_revenue
    from walmart_sales
    group by category
    order by total_revenue desc
    limit 1
),

high_stockout_store as (
    select
        store_location,
        round(
            sum(case when stockout_indicator = 1 then 1 else 0 end) / count(*) * 100,
            2
        ) as stockout_rate_percentage
    from walmart_sales
    group by store_location
    order by stockout_rate_percentage desc
    limit 1
),

forecast_problem_product as (
    select
        product_name,
        round(
            avg(abs(actual_demand - forecasted_demand) / nullif(actual_demand, 0) * 100),
            2
        ) as avg_forecast_error_percentage
    from walmart_sales
    group by product_name
    order by avg_forecast_error_percentage desc
    limit 1
),

supplier_risk as (
    select
        supplier_id,
        round(avg(supplier_lead_time), 2) as avg_supplier_lead_time,
        sum(case when stockout_indicator = 1 then 1 else 0 end) as stockout_count
    from walmart_sales
    group by supplier_id
    order by avg_supplier_lead_time desc, stockout_count desc
    limit 1
),

promotion_result as (
    select
        promotion_type,
        round(sum(quantity_sold * unit_price), 2) as total_revenue
    from walmart_sales
    where promotion_applied = 1
    group by promotion_type
    order by total_revenue desc
    limit 1
)

select
    1 as priority,
    concat('focus more inventory and marketing on ', category, ' category because it generates the highest revenue.') as recommendation,
    total_revenue as supporting_metric
from category_revenue

union all

select
    2 as priority,
    concat('reduce stockout risk at ', store_location, ' store because it has the highest stockout rate.') as recommendation,
    stockout_rate_percentage as supporting_metric
from high_stockout_store

union all

select
    3 as priority,
    concat('improve demand forecasting for ', product_name, ' because it has the highest forecast error.') as recommendation,
    avg_forecast_error_percentage as supporting_metric
from forecast_problem_product

union all

select
    4 as priority,
    concat('review supplier ', supplier_id, ' because it has high lead time and stockout count.') as recommendation,
    avg_supplier_lead_time as supporting_metric
from supplier_risk

union all

select
    5 as priority,
    concat('continue ', promotion_type, ' promotion because it generates the highest promotional revenue.') as recommendation,
    total_revenue as supporting_metric
from promotion_result

order by priority;
"""

result = pd.read_sql(query, engine)
result

,priority,recommendation,supporting_metric
0,1,focus more inventory and marketing on Electron...,7941631.80
1,2,"reduce stockout risk at New York, NY store bec...",55.42
2,3,improve demand forecasting for Laptop because ...,62.95
3,4,review supplier 321 because it has high lead t...,8.63
4,5,continue No Promotion promotion because it gen...,5439571.31


In [82]:
# check exact holiday column name
query = """
describe walmart_sales;
"""

result = pd.read_sql(query, engine)
result

,Field,Type,Null,Key,Default,Extra
0,transaction_id,bigint,YES,,None,
1,customer_id,bigint,YES,,None,
2,product_id,bigint,YES,,None,
3,product_name,text,YES,,None,
4,category,text,YES,,None,
5,quantity_sold,bigint,YES,,None,
6,unit_price,double,YES,,None,
7,transaction_date,datetime,YES,,None,
8,store_id,bigint,YES,,None,
9,store_location,text,YES,,None,


In [83]:
holiday_columns = result[result["Field"].str.contains("holiday", case=False, na=False)]
holiday_columns

,Field,Type,Null,Key,Default,Extra
23,holiday_indicator,tinyint(1),YES,,None,
